# Chatbot Expert AI Act — Version Agent LangGraph

Ce notebook reproduit l'application `app_agent.py` en version interactive.

**Différence fondamentale avec `chatbot_ai_act.ipynb` (version déterministe) :**
- Version déterministe : le routage est fait par du **code Python** (regex, score FAISS, détection de pronoms)
- Version agent : le **LLM décide lui-même** quel outil appeler via le mécanisme de **tool calling**

**Architecture agent :**
```
Question utilisateur
        ↓
   AGENT LangGraph (create_react_agent)
   Le LLM reçoit 5 outils et décide lequel utiliser :
        ├── recherche_article(num)        → docstore_lookup → texte intégral
        ├── recherche_considerant(num)    → docstore_lookup → texte intégral
        ├── recherche_annexe(num)         → docstore_lookup → texte intégral
        ├── recherche_ia_act(query)       → FAISS retriever → chunks pertinents
        └── recherche_web(query)          → DuckDuckGo (si connaissances insuffisantes)
        ↓
   Réponse rédigée par le LLM
```

**Modèle :** Llama 4 Scout 17B via Groq (gratuit, 30K tokens/min)

**Prérequis :** clé `GROQ_API_KEY` (gratuite sur https://console.groq.com)

---
## Cellule 1 — Installation des dépendances

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'])

---
## Cellule 2 — Imports

Imports clés par rapport à la version déterministe :
- `tool` : décorateur LangChain pour déclarer un outil accessible au LLM
- `create_react_agent` : crée un agent ReAct (Reasoning + Acting) avec boucle outil-LLM
- `ChatGroq` : client LLM Groq (pas d'Ollama, cloud uniquement)

In [ ]:
import os
import re
from pathlib import Path

# LangChain core
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.documents import Document

# Spécifique à la version agent
from langchain_core.tools import tool           # Décorateur @tool
from langchain_groq import ChatGroq              # Client LLM Groq
from langchain_community.tools import DuckDuckGoSearchRun  # Recherche web
from langgraph.prebuilt import create_react_agent  # Agent ReAct LangGraph

print('Imports OK')
print(f'  - langchain_core.tools.tool : décorateur pour déclarer un outil')
print(f'  - langgraph.prebuilt.create_react_agent : crée un agent avec boucle outil-LLM')
print(f'  - ChatGroq : client Groq (Llama 4 Scout 17B)')

---
## Cellule 3 — Configuration

**Différence avec la version déterministe :**
- Pas de `OLLAMA_MODEL` → cloud uniquement (Groq)
- `GROQ_MODEL` = `meta-llama/llama-4-scout-17b-16e-instruct` → 30K tokens/min (5× plus que Llama 3.1 8B)
- La clé `GROQ_API_KEY` doit être définie dans l'environnement

In [ ]:
# --- CONFIGURATION ---
MD_FILE          = Path('OJ_L_202401689_FR_TXTavec annexes.md')
INDEX_DIR        = Path('faiss_index')
MODEL_NAME       = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'
TOP_K            = 8
SCORE_THRESHOLD  = 0.35

# Modèle LLM — Groq uniquement (cloud)
GROQ_MODEL       = 'meta-llama/llama-4-scout-17b-16e-instruct'

# Clé API Groq (à définir)
# Option 1 : variable d'environnement
# os.environ['GROQ_API_KEY'] = 'gsk_votre_clé_ici'
# Option 2 : la clé est déjà dans l'environnement

groq_api_key = os.environ.get('GROQ_API_KEY', '')
if not groq_api_key:
    print('⚠  GROQ_API_KEY non définie !')
    print('   Décommentez la ligne os.environ ci-dessus et collez votre clé.')
    print('   Clé gratuite sur : https://console.groq.com')
else:
    print(f'✓ GROQ_API_KEY définie (commence par {groq_api_key[:8]}...)')

print(f'\nFichier source : {MD_FILE} ({"existe" if MD_FILE.exists() else "INTROUVABLE"})')
print(f'Index FAISS    : {INDEX_DIR} ({"existe" if INDEX_DIR.exists() else "à construire"})')
print(f'Modèle LLM    : {GROQ_MODEL}')

---
## Cellule 4 — Chargement de l'index FAISS

L'index FAISS est le même que pour la version déterministe :
- 796 chunks (236 considérants + 465 articles + 95 annexes)
- Vecteurs de dimension 768 (paraphrase-multilingual-mpnet-base-v2)
- Fichiers : `index.faiss` (2.4 Mo) + `index.pkl` (924 Ko)

Si l'index n'existe pas, exécutez d'abord `python build_index.py`.

In [ ]:
# Charger le modèle d'embeddings
print(f'Chargement du modèle d\'embeddings : {MODEL_NAME}')
embeddings = HuggingFaceEmbeddings(
    model_name=MODEL_NAME,
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True, 'batch_size': 32},
)
print('Modèle d\'embeddings chargé ✓')

# Charger l'index FAISS
if INDEX_DIR.exists():
    print('Index FAISS existant → rechargement...')
    db = FAISS.load_local(str(INDEX_DIR), embeddings, allow_dangerous_deserialization=True)
else:
    raise FileNotFoundError(
        f'Index FAISS introuvable dans {INDEX_DIR}/. '
        f'Exécutez : python build_index.py'
    )

print(f'Index FAISS prêt ✓ ({db.index.ntotal} vecteurs de dimension {db.index.d})')

# Créer le retriever pour la recherche sémantique
retriever = db.as_retriever(
    search_type='similarity_score_threshold',
    search_kwargs={'k': TOP_K, 'score_threshold': SCORE_THRESHOLD},
)
print(f'Retriever configuré ✓ (top-{TOP_K}, seuil {SCORE_THRESHOLD})')

---
## Cellule 5 — Connexion au LLM (Groq)

On utilise `ChatGroq` directement — pas de détection Ollama.

**Pourquoi Llama 4 Scout 17B et pas Llama 3.1 8B ?**
- L'agent fait **plusieurs appels LLM** par question (décision + outils + rédaction)
- Llama 3.1 8B = 6K tokens/min → quota dépassé en 2-3 questions
- Llama 4 Scout 17B = **30K tokens/min** (5× plus) → suffisant pour un agent

In [ ]:
llm = ChatGroq(
    model=GROQ_MODEL,
    api_key=groq_api_key,
    temperature=0.1,  # Quasi-déterministe (précision > créativité)
)
print(f'LLM connecté ✓ : {GROQ_MODEL} via Groq')

# Test rapide
try:
    test = llm.invoke('Réponds OK.')
    print(f'Test : {test.content[:50]}')
except Exception as e:
    print(f'ERREUR : {e}')

---
## Cellule 6 — Fonctions de base

Ces fonctions sont identiques à la version déterministe (`app.py`).
Elles sont utilisées par les outils de l'agent.

In [ ]:
def docstore_lookup(db, filters):
    """Recherche exacte dans le docstore FAISS par métadonnées.
    
    Parcourt TOUS les documents et filtre par critères.
    Ex: docstore_lookup(db, {"article": "5"}) → tous les chunks de l'article 5.
    Ce n'est PAS une recherche vectorielle — c'est un filtre sur les métadonnées.
    """
    results = []
    for doc_id in db.index_to_docstore_id.values():
        doc = db.docstore.search(doc_id)
        if all(doc.metadata.get(k) == v for k, v in filters.items()):
            results.append(doc)
    results.sort(key=lambda d: int(d.metadata.get('paragraph', '0') or '0'))
    return results


def format_docs(docs):
    """Concatène les contenus des documents avec un séparateur."""
    return '\n\n---\n\n'.join(doc.page_content for doc in docs)


def get_sources(docs):
    """Extrait les labels de sources lisibles."""
    sources = []
    for doc in docs:
        m = doc.metadata
        if m.get('type') == 'article':
            label = f"Article {m['article']} : {m['title']}"
            if m.get('chapter'):
                label = f"Chapitre {m['chapter']} > {label}"
        elif m.get('type') == 'annexe':
            label = f"Annexe {m['annexe']} : {m['title']}"
        else:
            label = m.get('title', 'Considérant')
        sources.append(label)
    return sources


print('Fonctions de base définies ✓')

---
## Cellule 7 — Définition des 5 outils (@tool)

**C'est ici que réside la différence fondamentale avec la version déterministe.**

Dans `app.py`, le code Python décide quel traitement appliquer (regex → docstore, FAISS → LLM, etc.).

Ici, on **déclare des outils** avec le décorateur `@tool` de LangChain. Le LLM lit les **docstrings** de chaque outil et décide **lui-même** lequel appeler en fonction de la question.

**Anatomie d'un outil :**
```python
@tool
def recherche_article(article_num: str) -> str:
    \"\"\"Description lue par le LLM pour décider quand appeler cet outil.
    
    Args:
        article_num: description du paramètre (le LLM l'utilise pour passer la bonne valeur)
    \"\"\"
    # Code exécuté quand le LLM appelle l'outil
    return "résultat textuel renvoyé au LLM"
```

**Points critiques :**
- La **docstring** est le contrat entre le développeur et le LLM
- Le **type des arguments** (`str`) est utilisé pour la sérialisation JSON
- Le **retour** doit être une chaîne (le LLM ne lit que du texte)

In [ ]:
# Variable globale pour collecter les sources (pas de st.session_state en notebook)
collected_sources = []


@tool
def recherche_article(article_num: str) -> str:
    """Recherche le texte intégral d'un article du AI Act par son numéro.
    Utilise cet outil quand l'utilisateur demande un article précis.

    Args:
        article_num: numéro de l'article sous forme de chaîne (ex: "5", "66", "1")
    """
    num = '1' if article_num.lower().strip() in ('premier', '1er') else article_num.strip()
    docs = docstore_lookup(db, {'article': num})
    if not docs:
        return f'Article {num} non trouvé dans la base AI Act.'
    collected_sources.extend(get_sources(docs))
    return format_docs(docs)


@tool
def recherche_considerant(numero: str) -> str:
    """Recherche le texte d'un considérant du AI Act par son numéro.
    Utilise cet outil quand l'utilisateur demande un considérant précis.

    Args:
        numero: numéro du considérant sous forme de chaîne (ex: "12", "55")
    """
    num = numero.strip().strip('()')
    docs = docstore_lookup(db, {'type': 'considerant', 'numero': f'({num})'})
    if not docs:
        return f'Considérant {num} non trouvé dans la base AI Act.'
    collected_sources.extend(get_sources(docs))
    return format_docs(docs)


@tool
def recherche_annexe(annexe_num: str) -> str:
    """Recherche le texte intégral d'une annexe du AI Act par son numéro romain.
    Utilise cet outil quand l'utilisateur demande une annexe.

    Args:
        annexe_num: numéro de l'annexe en chiffres romains (ex: "III", "XI", "I")
    """
    num = annexe_num.strip().upper()
    docs = docstore_lookup(db, {'type': 'annexe', 'annexe': num})
    if not docs:
        return f'Annexe {num} non trouvée dans la base AI Act.'
    collected_sources.extend(get_sources(docs))
    return format_docs(docs)


@tool
def recherche_ia_act(query: str) -> str:
    """Recherche sémantique dans le AI Act pour les questions générales.
    Utilise cet outil pour les questions sur les obligations, interdictions,
    sanctions, conformité, etc. — tout ce qui n'est pas un numéro d'article précis.

    Args:
        query: la question ou les mots-clés à rechercher dans le AI Act
    """
    docs = retriever.invoke(query)
    if not docs:
        return 'Aucun passage pertinent trouvé dans le AI Act.'
    collected_sources.extend(get_sources(docs))
    source_list = '\n'.join(f'- {s}' for s in get_sources(docs))
    context = format_docs(docs)
    if len(context) > 4000:
        context = context[:4000] + '\n\n[... tronqué ...]'
    return f'Sources AI Act :\n{source_list}\n\nExtraits :\n{context}'


@tool
def recherche_web(query: str) -> str:
    """Recherche sur internet via DuckDuckGo.
    Utilise cet outil UNIQUEMENT si les 4 autres outils n'ont pas trouvé
    l'information ET que tes propres connaissances ne suffisent pas.

    Args:
        query: la question à rechercher sur internet
    """
    search = DuckDuckGoSearchRun()
    parts = []
    try:
        parts.append(search.invoke(query))
    except Exception:
        pass
    try:
        parts.append(search.invoke(query + ' results 2025'))
    except Exception:
        pass
    web = '\n\n'.join(p for p in parts if p)
    if not web or len(web) < 50:
        return 'Aucun résultat pertinent sur internet.'
    collected_sources.append('Recherche internet (DuckDuckGo)')
    return f'Résultats internet :\n\n{web[:2500]}'


tools = [recherche_article, recherche_considerant, recherche_annexe, recherche_ia_act, recherche_web]
print(f'5 outils définis ✓')
for t in tools:
    print(f'  - {t.name} : {t.description[:80]}...')

---
## Cellule 8 — Prompt système de l'agent

Le prompt est **différent** de celui de `app.py` :
- Il **décrit les outils** et quand les utiliser (le LLM en a besoin pour décider)
- Il n'a **pas de règles de routage par code Python** (c'est le LLM qui route, pas du code Python)
- Il garde les règles de **qualité de réponse** et de **mémoire**

**Point clé :** dans un agent, le prompt système est encore plus important que dans un chatbot simple,
car il guide les décisions de tool calling. Une mauvaise description d'outil → mauvais appels.

In [ ]:
AGENT_PROMPT = """\
Tu es un assistant expert du Règlement européen sur l'Intelligence Artificielle \
(AI Act, Règlement UE 2024/1689). Réponds en français.

Tu as 5 outils à ta disposition :
- recherche_article : retourne le texte intégral d'un article par son numéro (ex: "5", "66")
- recherche_considerant : retourne le texte d'un considérant par son numéro (ex: "12", "55")
- recherche_annexe : retourne le texte d'une annexe par son numéro en chiffres romains (ex: "III", "XI")
- recherche_ia_act : recherche sémantique dans tout le AI Act pour les questions générales
- recherche_web : recherche sur internet via DuckDuckGo

RÈGLES D'UTILISATION DES OUTILS :
1. Quand l'utilisateur demande un article précis, utilise recherche_article.
2. Quand l'utilisateur demande un considérant, utilise recherche_considerant.
3. Quand l'utilisateur demande une annexe, utilise recherche_annexe.
4. Pour les questions générales sur le AI Act, utilise recherche_ia_act.
5. Si l'utilisateur demande PLUSIEURS articles, appelle recherche_article PLUSIEURS FOIS.
6. Si la question ne concerne PAS le AI Act, cherche d'abord dans tes
   connaissances propres. Si suffisant, réponds directement.
7. Si tes connaissances ne suffisent PAS, utilise recherche_web.
   PRÉCISE TOUJOURS que l'info vient d'internet.

RÈGLES DE RÉPONSE :
1. Cite les articles et considérants exacts.
2. Liste les obligations et interdictions.
3. Structure avec titres et puces.
4. Si l'info vient d'internet (recherche_web), INDIQUE-LE clairement.
5. Utilise l'historique pour les questions de suivi.
6. Si résumé demandé, fournis un résumé synthétique.
"""

print(f'Prompt système défini ✓ ({len(AGENT_PROMPT)} caractères)')

---
## Cellule 9 — Création de l'agent LangGraph

**`create_react_agent`** crée un agent **ReAct** (Reasoning + Acting) :

```
Boucle ReAct :
1. Le LLM reçoit la question + les outils disponibles
2. Il décide : appeler un outil OU répondre directement
3. Si outil appelé → le résultat est renvoyé au LLM
4. Retour à l'étape 2 (le LLM peut appeler un autre outil)
5. Quand le LLM décide de répondre → fin de la boucle
```

**Différence avec un simple `llm.invoke()` :**
- `llm.invoke()` = 1 appel, 1 réponse textuelle
- `agent.invoke()` = N appels, le LLM orchestre les outils en boucle

In [ ]:
agent = create_react_agent(
    model=llm,
    tools=tools,
)

print('Agent LangGraph créé ✓')
print(f'  Modèle : {GROQ_MODEL}')
print(f'  Outils : {[t.name for t in tools]}')
print(f'  Type   : ReAct (Reasoning + Acting)')

---
## Cellule 10 — Mémoire conversationnelle et fonction ask()

La mémoire est gérée manuellement (même principe que `app.py`) :
- `InMemoryChatMessageHistory` stocke les échanges
- Les 6 derniers messages sont envoyés à l'agent à chaque appel
- Les messages longs sont tronqués à 800 caractères

**Différence avec `app.py` :**
- Pas de `process_question()` avec regex/FAISS/pronoms
- Un seul appel : `agent.invoke({"messages": [...]})`
- L'agent décide tout seul

In [ ]:
# Mémoire conversationnelle
chat_history = InMemoryChatMessageHistory()


def ask(question):
    """Envoie une question à l'agent et affiche la réponse."""
    global collected_sources
    collected_sources = []  # Réinitialiser les sources
    
    print(f'\nQuestion : {question}')
    print('=' * 70)
    
    # Construire les messages : prompt + historique + question
    messages = [SystemMessage(content=AGENT_PROMPT)]
    
    # Ajouter l'historique (6 derniers messages, tronqués à 800 chars)
    if chat_history.messages:
        for msg in chat_history.messages[-6:]:
            content = msg.content[:800] + '...' if len(msg.content) > 800 else msg.content
            messages.append(type(msg)(content=content))
    
    # Ajouter la question
    messages.append(HumanMessage(content=question))
    
    # Invoquer l'agent
    try:
        result = agent.invoke({'messages': messages})
    except Exception as e:
        print(f'ERREUR : {e}')
        return str(e)
    
    # Extraire la réponse finale
    response = result['messages'][-1].content
    
    # Extraire les outils appelés
    tools_called = []
    for msg in result['messages']:
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            for tc in msg.tool_calls:
                tools_called.append(f"{tc['name']}({tc['args']})")
    
    # Afficher les outils appelés
    if tools_called:
        print(f'🔧 Outils appelés ({len(tools_called)}) :')
        for tc in tools_called:
            print(f'   → {tc}')
        print('-' * 70)
    else:
        print('💡 Réponse directe (aucun outil appelé)')
        print('-' * 70)
    
    # Afficher la réponse
    print(response)
    
    # Afficher les sources
    sources = list(dict.fromkeys(collected_sources))
    if sources:
        print(f'\n📚 Sources ({len(sources)}) :')
        for s in sources:
            print(f'   - {s}')
    
    # Sauvegarder dans la mémoire
    chat_history.add_message(HumanMessage(content=question))
    chat_history.add_message(AIMessage(content=response))
    
    return response


print('Fonction ask() prête ✓')
print('Mémoire initialisée ✓')

---
## Tests

### Test 1 — Article précis
L'agent devrait appeler `recherche_article("5")`.

In [ ]:
ask("Donne-moi l'article 5")

### Test 2 — Multi-articles
L'agent devrait appeler `recherche_article` **deux fois** : une pour le 5, une pour le 8.

In [ ]:
ask("Articles 5 et 8")

### Test 3 — Résumé (l'agent résume au lieu de retourner le texte brut)
L'agent devrait appeler `recherche_article("66")` puis résumer dans sa réponse.

In [ ]:
ask("Résume l'article 66")

### Test 4 — Recherche sémantique (RAG)
L'agent devrait appeler `recherche_ia_act` avec une requête sémantique.

In [ ]:
ask("Je recrute par IA, suis-je conforme ?")

### Test 5 — Annexe
L'agent devrait appeler `recherche_annexe("III")`.

In [ ]:
ask("Donne l'annexe III")

### Test 6 — Culture générale (pas d'outil)
L'agent devrait répondre **directement** sans appeler d'outil.

In [ ]:
ask("Qui est Victor Hugo ?")

### Test 7 — Mémoire conversationnelle
L'agent devrait utiliser l'historique pour comprendre que « il » = Victor Hugo.

In [ ]:
ask("Était-il un peintre aussi ?")

---
## Vérifier la mémoire

In [ ]:
print(f'Mémoire : {len(chat_history.messages)} messages\n')
for m in chat_history.messages:
    role = 'USER' if isinstance(m, HumanMessage) else 'AI'
    print(f'  [{role:4s}] {m.content[:80]}')
    print()

---
## Mode interactif
Tapez vos questions en boucle. Entrez `quit` pour arrêter.

In [ ]:
print('Agent AI Act — Tapez \'quit\' pour quitter')
print('=' * 50)

while True:
    question = input('\nVotre question : ')
    if question.strip().lower() in ('quit', 'exit', 'q'):
        print('Au revoir !')
        break
    if not question.strip():
        continue
    ask(question)